# Global (whole-population) region A/B mean |SHAP|, next to the local band, with growth ratios

Standalone notebook, deliberately kept separate from `shap_did/04_02_shap_did_concentration.ipynb`
so running it does not also re-run that notebook's DiD/dose/drift/shared-claims cells — this only
needs a parquet load and a groupby-mean, nothing SHAP gets recomputed here. `chosen_h` is the one
exception: it is a cheap cell-count grid search (`ShapDiDEstimator.select_local_h`), re-derived
here rather than read off a file, so this notebook has no dependency on `shap_did/04_02` having
been (re)run first, and never hardcodes the $h$ literal.

For v2(test) and v3(oot) only (`config.OOT_SPLIT` — the same two splits Table~\ref{tab:shapdid-estimate}
uses), on the same whole-population $n=5{,}000$ explain sample. Writes, per (version, split):

- **`<v>_<split>_region_attributions.csv`** — the per-claim, region-tagged attribution table
  (`claim_id` kept via `subdir="internal"`), so a later question does not need to reload and
  re-merge `version_pair_table()` from scratch. Real-named and aliased twins, both under
  `.../internal/`.
- **`<v>_<split>_global_region_mean_shap.csv`** — mean |SHAP| per feature: region A, region B, and
  `ALL` (no region filter — a reference figure, identical to `VersionData.mean_abs_shap` /
  `00_shap_attribution.ipynb`'s `01a_real_shap_global_importance_*`; not used as a ratio
  denominator below). A and B are filtered by `region` only (no era, no local-band restriction) —
  the identical filter `ShapDiDEstimator.estimate()` uses for `tab:shapdid-estimate`'s $S_A$/$S_B$.
- **`<v>_<split>_global_local_region_ratio.csv`** — every feature, global A/B/ALL next to local
  A'/B' (same $h$ `tab:shapdid-local` uses) and how many times EACH REGION's mean |SHAP| grew from
  its own global figure to its local one (`local_A/global_A`, `local_B/global_B` — A compared
  against A, B against B, never against `ALL`). For v3(oot) at this $h$, region B's local band is
  the *entire* global B population (`n_local == n_global`, see the SATURATED print in the loop
      below), so `B_ratio_local_over_global` is trivially $\approx 1$ there — a real, documented
  property of this version/h combination, not a computation error.
- **`<v>_<split>_top20_by_global_B_ratio_ordered.csv`** — the dissertation-table-ready cut: the
  top 20 features by GLOBAL region B (the model-scrapped population), reordered by how much each
  one's B grew locally (`B_ratio_local_over_global`, descending) — the "which globally-relevant
  feature jumps hardest right at τ" ordering.

All four, real-named under `real_named/`, aliased twins under `alias/` (per-claim ones additionally
nested under `.../internal/`, both sides).


In [ ]:
# §0 -- setup (analysis .venv kernel)
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import figstyle
import feature_alias
from estimator import concentration
from estimator.effect.shap_did import (
    ShapDiDEstimator, version_pair_table, cell_mabs, local_region, ID_COL, TAG_COLS,
)

figstyle.FIG_DIR = ROOT / "figures" / "addtional"
figstyle.ALIAS_SPLIT = True  # real-name/alias per-feature outputs -> real_named/ / alias/
pd.set_option("display.width", 160)
est = ShapDiDEstimator()
print("ROOT =", ROOT)
print("FIG_DIR =", figstyle.FIG_DIR)


## v2(test) and v3(oot) only

Both splits come from `config.OOT_SPLIT`, never hardcoded, so this stays locked to whatever
`tab:shapdid-estimate` itself uses if that mapping ever changes.


In [ ]:
# shared h for the v2(test)->v3(oot) pair, exactly as shap_did/04_02's own §2b picks it --
# never hardcoded: re-derived here so this notebook has no file dependency on that notebook.
chosen_h, _ = est.select_local_h("v2", config.OOT_SPLIT["v2"], "v3", config.OOT_SPLIT["v3"])
print("chosen_h =", chosen_h)


In [ ]:
NON_FEATURE_COLS = [ID_COL] + TAG_COLS + [concentration.BASE_COL]
VERSION_SPLITS = [("v2", config.OOT_SPLIT["v2"]), ("v3", config.OOT_SPLIT["v3"])]
TOP_N_GLOBAL_B = 20

for version, split in VERSION_SPLITS:
    table = version_pair_table(version, split)  # parquet load + merge only -- no SHAP recomputed
    feature_cols = [c for c in table.columns if c not in NON_FEATURE_COLS]
    print(f"[{version}/{split}] {len(table):,} claims, {len(feature_cols)} features "
          f"(region A={int((table['region'] == 'A').sum())}, "
          f"B={int((table['region'] == 'B').sum())})")

    # --- per-claim, region-tagged attributions -- kept for reuse, claim_id via subdir="internal" ---
    figstyle.save_table(table, f"{version}_{split}_region_attributions",
                         index=False, subdir="internal")
    alias_table = table.rename(
        columns=dict(zip(feature_cols, feature_alias.to_alias(version, feature_cols))))
    figstyle.save_table(alias_table, f"alias_{version}_{split}_region_attributions",
                         index=False, subdir="internal")

    # --- global (whole-population) mean|SHAP| by region -- same region-only filter estimate() uses ---
    wide = pd.DataFrame({r: cell_mabs(table, region=r) for r in ("A", "B")}).sort_values(
        "B", ascending=False)
    # ALL: no region filter at all -- the SAME quantity as VersionData.mean_abs_shap /
    # 00_shap_attribution.ipynb's `01a_real_shap_global_importance_*` figure, since both read the
    # identical attributions parquet via loaders.load(version, split).attributions. Kept as a
    # reference/sanity-check column only -- the ratio columns below compare A-to-A and B-to-B, not
    # against this.
    wide["ALL"] = cell_mabs(table)
    wide.index.name = "feature"
    figstyle.save_table(wide, f"{version}_{split}_global_region_mean_shap")
    alias_wide = wide.set_axis(
        feature_alias.to_alias(version, list(wide.index)), axis=0).rename_axis("feature")
    figstyle.save_table(alias_wide, f"alias_{version}_{split}_global_region_mean_shap")

    # --- local (near-tau, h=chosen_h) mean|SHAP| by region -- same restriction tab:shapdid-local uses ---
    band = local_region(table, version, chosen_h)
    wide_local = pd.DataFrame(
        {r: cell_mabs(band, drop_extra=["region_local"], region_local=r) for r in ("A", "B")})

    # --- diagnostic: local band should be a STRICT SUBSET of the global population per region.
    # if n_local == n_global for a region, local_* and global_* are computed on the identical rows
    # (band saturated -- e.g. tau within h of the score ceiling), so the ratio column for that
    # region is trivially ~1 and not a real "local vs global" comparison for this version. ---
    for r in ("A", "B"):
        n_global = int((table["region"] == r).sum())
        n_local = int((band["region_local"] == r).sum())
        flag = "  <-- SATURATED: local == global, ratio is trivial" if n_local == n_global else ""
        print(f"  region {r}: n_global={n_global:,}  n_local={n_local:,}{flag}")

    # --- merge: global A/B next to local A'/B', plus local/global growth ratio per region ---
    ratio = pd.DataFrame({
        "global_A": wide["A"], "global_B": wide["B"], "global_ALL": wide["ALL"],
        "local_A": wide_local["A"], "local_B": wide_local["B"],
    })
    ratio["A_ratio_local_over_global"] = ratio["local_A"] / ratio["global_A"]
    ratio["B_ratio_local_over_global"] = ratio["local_B"] / ratio["global_B"]
    ratio.index.name = "feature"
    ratio = ratio.sort_values("global_B", ascending=False)
    figstyle.save_table(ratio, f"{version}_{split}_global_local_region_ratio")
    alias_ratio = ratio.set_axis(
        feature_alias.to_alias(version, list(ratio.index)), axis=0).rename_axis("feature")
    figstyle.save_table(alias_ratio, f"alias_{version}_{split}_global_local_region_ratio")

    # --- dissertation-table CSV: top-N by GLOBAL B, reordered by how much B grew locally ---
    top_n = ratio.head(TOP_N_GLOBAL_B)  # already sorted by global_B above
    top_n_ordered = top_n.sort_values("B_ratio_local_over_global", ascending=False)
    figstyle.save_table(top_n_ordered, f"{version}_{split}_top{TOP_N_GLOBAL_B}_by_global_B_ratio_ordered")
    alias_top_n_ordered = top_n_ordered.set_axis(
        feature_alias.to_alias(version, list(top_n_ordered.index)), axis=0).rename_axis("feature")
    figstyle.save_table(alias_top_n_ordered,
                         f"alias_{version}_{split}_top{TOP_N_GLOBAL_B}_by_global_B_ratio_ordered")

print("done")
